# SGIP Method Walkthrough (Pipeline Only)
This notebook walks through the exact algorithm in `method_v2.md` using the current codebase. Each step has a markdown explanation, executable code, and visualization.

## Step 0 — Imports and Helper Functions

In [ ]:
import json
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

from configs.pipeline_config import load_pipeline_config
from datasets.dataset import load_dataset
from image_processings.image_pre_seg import image_i_segment
from image_processings.info import Info
from debug_tests.run_tta import run_segmentation_with_info
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

plt.rcParams['figure.dpi'] = 120

MAIN_DIR = Path('.').resolve()

# --- Visualization helpers ---
def _to_uint8(img):
    if img.dtype == np.uint8:
        return img
    if img.max() <= 1.0:
        return (img * 255).astype(np.uint8)
    return img.astype(np.uint8)

def _overlay(image, mask, color=(0, 255, 0), alpha=0.4):
    base = _to_uint8(image)
    if base.ndim == 2:
        base = np.repeat(base[..., None], 3, axis=2)
    overlay = base.copy()
    overlay[mask] = color
    return (base * (1 - alpha) + overlay * alpha).astype(np.uint8)

def _draw_points(ax, points, labels):
    if points is None or len(points) == 0:
        return
    pts = np.asarray(points)
    lbl = np.asarray(labels)
    pos = pts[lbl == 1]
    neg = pts[lbl == 0]
    if len(pos):
        ax.scatter(pos[:, 0], pos[:, 1], c='lime', s=20, edgecolors='k', linewidths=0.3)
    if len(neg):
        ax.scatter(neg[:, 0], neg[:, 1], c='red', s=20, edgecolors='k', linewidths=0.3)

def _show_heatmap(ax, heat, title):
    ax.imshow(heat, cmap='magma')
    ax.set_title(title)
    ax.axis('off')


## Step 1 — Load Config, Model, and One Sample

In [ ]:
# Load pipeline config
pipeline_cfg = load_pipeline_config(Path('configs/pipeline.json'))

# Load constants for model paths
constants = json.loads(Path('CONSTANT.json').read_text(encoding='utf-8'))

# Build SAM2 predictor
model = build_sam2(constants['model_cfg'], constants['checkpoint'], device='cuda' if torch.cuda.is_available() else 'cpu')
predictor = SAM2ImagePredictor(model)
predictor.model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Load dataset and pick one sample
images, gt_masks, names = load_dataset(
    pipeline_cfg.dataset.name,
    target_long_edge=pipeline_cfg.dataset.target_long_edge,
    return_paths=True,
)

idx = 0
image = images[idx]
gt_mask = gt_masks[idx]
name = names[idx]

plt.figure(figsize=(4, 4))
plt.imshow(_to_uint8(image))
plt.title(f'Input image: {name}')
plt.axis('off')
plt.show()


## Step 2 — Preprocess (Resize + SLIC)

In [ ]:
pre_segment = image_i_segment(
    image=image,
    new_size_of_image=pipeline_cfg.preprocessing.image_size,
    num_node_for_graph=pipeline_cfg.preprocessing.num_graph_nodes,
    compactness_in_SLIC=pipeline_cfg.preprocessing.slic.compactness,
    sigma_in_SLIC=pipeline_cfg.preprocessing.slic.sigma,
    min_size_factor_in_SLIC=pipeline_cfg.preprocessing.slic.min_size_factor,
    max_size_factor_in_SLIC=pipeline_cfg.preprocessing.slic.max_size_factor,
)

img_resized = pre_segment.image_resized
segments = np.array(pre_segment.segment_without_padding)

plt.figure(figsize=(4, 4))
plt.imshow(_to_uint8(img_resized))
plt.title('Resized image')
plt.axis('off')
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(segments, cmap='tab20')
plt.title('SLIC superpixels')
plt.axis('off')
plt.show()


## Step 3 — Initialize Prompts

In [ ]:
info = Info(
    segment=segments,
    logits=None,
    image=_to_uint8(img_resized),
    graph=pre_segment.graph,
    settings=pipeline_cfg.algorithm,
    debug_mode=False,
    mask_prompt_source=pipeline_cfg.sam.mask_prompt_source,
)

initial_bundle = info.build_initial_prompts()

plt.figure(figsize=(4, 4))
plt.imshow(_to_uint8(img_resized))
_draw_points(plt.gca(), initial_bundle.points, initial_bundle.labels)
plt.title('Initial prompts')
plt.axis('off')
plt.show()


## Step 4 — Run Full Pipeline (to collect history)

In [ ]:
base_mask, history, vis_image, segments, info = run_segmentation_with_info(
    image, pipeline_cfg, predictor
)
print(f'Total steps: {len(history)}')


## Step 5 — Visualize Each Iteration (Prompts / Logits / Mask / Prompt Mask)

In [ ]:
max_steps = None  # set a number to limit
steps = history if max_steps is None else history[:max_steps]

num_steps = len(steps)
fig, axes = plt.subplots(num_steps, 4, figsize=(16, 4 * num_steps))
if num_steps == 1:
    axes = np.expand_dims(axes, axis=0)

for row, step in enumerate(steps):
    # Image + prompts
    axes[row, 0].imshow(_to_uint8(vis_image))
    _draw_points(axes[row, 0], step.prompts.points, step.prompts.labels)
    axes[row, 0].set_title(f'step {row}: prompts')
    axes[row, 0].axis('off')

    # Logits heatmap
    _show_heatmap(axes[row, 1], step.logits, f'step {row}: logits')

    # Mask overlay
    axes[row, 2].imshow(_overlay(vis_image, step.mask))
    axes[row, 2].set_title(f'step {row}: mask')
    axes[row, 2].axis('off')

    # Mask prompt (if any)
    mask_prompt = step.prompts.mask_prompt
    if mask_prompt is not None:
        _show_heatmap(axes[row, 3], mask_prompt, f'step {row}: mask_prompt')
    else:
        axes[row, 3].text(0.5, 0.5, 'None', ha='center', va='center')
        axes[row, 3].set_title(f'step {row}: mask_prompt')
        axes[row, 3].axis('off')

plt.tight_layout()
plt.show()


## Step 6 — Final Selection Result

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(_overlay(vis_image, base_mask))
plt.title('Final selected mask')
plt.axis('off')
plt.show()
